In [ ]:
#IMPORTS

import random
import json
from datetime import datetime, timedelta
from faker import Faker
import csv
import re

#configurações iniciais
num_leads = 256
id_inicial = 1056

#dados auxiliares
faker = Faker('pt_BR')
ceps_amostrados = random.sample(ceps_data, num_leads)
with open("cepsbrasil.json", encoding='utf-8') as f: ceps_data = json.load(f)
with open("ddds.json", encoding="utf-8") as f:
    ddd_por_estado = json.load(f)



In [82]:
# Geração de CPF válido
def gera_cpf():
    def calcula_digito(digs):
        s = sum([v * (len(digs) + 1 - i) for i, v in enumerate(digs)])
        res = 11 - s % 11
        return res if res < 10 else 0
    digs = [random.randint(0, 9) for _ in range(9)]
    digs.append(calcula_digito(digs))
    digs.append(calcula_digito(digs))
    return ''.join(map(str, digs))

In [83]:
#cria telefones
def get_ddd_por_uf(uf):
    uf = uf.upper()
    for item in ddd_por_estado:
        if item["uf"] == uf:
            return item
    return None

def gera_telefone_ponderado(cep, uf):
    # cep não usado aqui, pode remover do parâmetro se quiser
    dados_ddd = get_ddd_por_uf(uf)

    if not dados_ddd:
        ddd = '11'  # Fallback São Paulo
    else:
        usa_vizinho = random.random() < 0.25
        ddds = dados_ddd['alternativos'] if usa_vizinho and dados_ddd.get('alternativos') else dados_ddd['principais']
        ddd = random.choice(ddds)

    # Decide se é celular ou fixo (80% celular)
    if random.random() < 0.8:
        prefixo = '9' + str(random.randint(0, 9999)).zfill(4)
    else:
        prefixo = str(random.randint(3000, 5999)).zfill(4)

    sufixo = str(random.randint(0, 9999)).zfill(4)

    return f"+55 ({ddd}) {prefixo}-{sufixo}"


In [84]:
# Data de nascimento entre 18 e 73 anos até 05/08/2025
def gera_data_nascimento():
    hoje = datetime(2025, 8, 5)
    idade_min = 18
    idade_max = 73
    dias_min = idade_min * 365
    dias_max = idade_max * 365
    nascimento = hoje - timedelta(days=random.randint(dias_min, dias_max))
    return nascimento.strftime('%Y-%m-%d')

In [85]:
#valida nomes
def validar_nome(nome):
    # Lista de padrões a remover
    padroes = [
        r'(excelent[íi]ssim[oa])', r'(ilustr[íi]ssim[oa])', r'(doutor[ae]?)', r'(honoris causa)',
        r'(dr\.?|dra\.?)', r'(prof\.?|professor[ae]?)', r'(mestre)', r'(ms\.?)', r'(ph\.?d)', r'(especialista)',
        r'(sr\.?|sra\.?|sr[ªa]?|senhor[ae]s?)', r'(senhorita)',
        r'(pe\.?|padre)', r'(pastor[ae]?)', r'(bispo)', r'(irm[ãa]?)', r'(mission[áa]rio)', r'(reverendo)',
        r'(capit[ãa]o)', r'(tenente)', r'(general)', r'(coronel)', r'(major)', r'(sargento)',
        r'(eng\.?|engenheiro)', r'(arquiteto)', r'(diretor)', r'(gerente)', r'(supervisor)', r'(ceo)',
        r'(analista)', r'(consultor)', r'(desembargador)', r'(ju[ií]z)', r'(advogado)', r'(promotor)',
        r'(vereador)', r'(deputado)', r'(senador)', r'(presidente)'
    ]

    # Junta tudo num único regex
    regex = r'^\s*(?:' + '|'.join(padroes) + r')\s+'

    # Remove título inicial repetidamente (caso haja mais de um)
    while re.match(regex, nome, re.IGNORECASE):
        nome = re.sub(regex, '', nome, flags=re.IGNORECASE)

    return nome.strip()

In [86]:
#gerar complemento
def gerar_complemento():
    tipo_imovel = random.choices(
        ['casa', 'apartamento', 'condominio'],
        weights=[0.3, 0.4, 0.3],  # Casas são menos frequentes em áreas urbanas
        k=1
    )[0]

    if tipo_imovel == 'casa':
        # 50% das casas não têm complemento
        return None if random.random() < 0.75 else f"Casa {random.randint(1, 99)}"

    elif tipo_imovel == 'apartamento':
        return f"Apto {random.randint(11, 999)}"

    elif tipo_imovel == 'condominio':
        bloco = random.choice(['A', 'B', 'C', 'D', 'E']) + str(random.randint(1, 5))
        apto = random.randint(11, 999)
        return f"Bloco {bloco}, Apto {apto}"

In [87]:
# Geração dos leads
leads = []
for i in range(num_leads):
    lead = ceps_amostrados[i]
    uf = lead.get('estado', '')
    leads.append({
        "id": str(id_inicial + i).zfill(7),
        "nome": validar_nome(faker.name()),
        "telefone": gera_telefone_ponderado(None, uf),  # pode passar None, cep não é usado
        "nascimento": gera_data_nascimento(),
        "cpf": gera_cpf(),
        "cep": str(lead.get('cep', '')).zfill(8),
        "uf": uf,
        "logradouro": lead.get('rua', ''),
        "bairro": lead.get('bairro', ''),
        "cidade": lead.get('cidade', ''),
        "complemento": gerar_complemento()
    })

In [88]:
# Salvando em CSV
with open("leads_gerados.csv", mode='w', encoding='utf-8', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=leads[0].keys())
    writer.writeheader()
    writer.writerows(leads)

print("Arquivo leads_gerados.csv criado com sucesso.")

Arquivo leads_gerados.csv criado com sucesso.
